### Übung 01: Numerische Lösung des Kepler-Problems in 2D

Wir betrachten einen leichten Körper der Masse $m$ im Gravitationsfeld einer sehr viel schwereren Zentralmasse $M$. Die Bewegung der Zentralmasse wird vernachlässigt. Die Bahn liegt in einer Ebene und wird in kartesischen Koordinaten durch

$$x(t),\qquad y(t)$$

beschrieben. In Einheiten mit $GM=1$ lautet die Newtonsche Bewegungsgleichung

$$\ddot x=-\frac{x}{(x^2+y^2)^{3/2}},\qquad \ddot y=-\frac{y}{(x^2+y^2)^{3/2}}.$$

Für die numerische Integration schreiben wir daraus vier Differentialgleichungen erster Ordnung:

$$\dot x=v_x,\qquad \dot y=v_y,\qquad \dot v_x=a_x(x,y),\qquad \dot v_y=a_y(x,y).$$

Die spezifische mechanische Energie, also die Energie pro Masse $m$, ist

$$E=\frac12(v_x^2+v_y^2)-\frac1{\sqrt{x^2+y^2}}.$$

Gebundene exakte Bahnen sind Ellipsen. Wir vergleichen drei numerische Verfahren hinsichtlich Bahnform, Genauigkeit und Energiefehler.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Beschleunigung und Euler-Verfahren

Das explizite Euler-Verfahren verwendet während eines Schritts die alten Geschwindigkeiten und die alte Beschleunigung. Es ist ein Verfahren erster Ordnung.

In [ ]:
def kepler_acceleration(x, y):
    radius = np.sqrt(x**2 + y**2)
    ax = -x / radius**3
    ay = -y / radius**3
    return ax, ay

def mechanical_energy(x, y, vx, vy):
    radius = np.sqrt(x**2 + y**2)
    return 0.5*(vx**2 + vy**2) - 1.0/radius

def euler_step(x, y, vx, vy, h, acceleration=kepler_acceleration):
    ax, ay = acceleration(x, y)
    x_new = x + h*vx
    y_new = y + h*vy
    vx_new = vx + h*ax
    vy_new = vy + h*ay
    return x_new, y_new, vx_new, vy_new

def simulate(step, x0, y0, vx0, vy0, h, n_steps,
             acceleration=kepler_acceleration):
    x = np.empty(n_steps + 1)
    y = np.empty(n_steps + 1)
    vx = np.empty(n_steps + 1)
    vy = np.empty(n_steps + 1)
    x[0], y[0], vx[0], vy[0] = x0, y0, vx0, vy0
    for n in range(n_steps):
        x[n + 1], y[n + 1], vx[n + 1], vy[n + 1] = step(
            x[n], y[n], vx[n], vy[n], h, acceleration
        )
    t = h*np.arange(n_steps + 1)
    return t, x, y, vx, vy

## Aufgabe 1 – Velocity-Verlet

Implementieren Sie einen Velocity-Verlet-Schritt für $x,y,v_x,v_y$:

In [ ]:
def verlet_step(x, y, vx, vy, h, acceleration=kepler_acceleration):
    # TODO: Velocity-Verlet für x, y, vx und vy implementieren
    raise NotImplementedError

## Aufgabe 2 – Klassisches Runge-Kutta-Verfahren

Implementieren Sie das klassische Runge-Kutta-Verfahren vierter Ordnung (RK4). Achten Sie darauf, in jeder Zwischenstufe sowohl Ort als auch Geschwindigkeit konsistent zu aktualisieren.

In [ ]:
def rk4_step(x, y, vx, vy, h, acceleration=kepler_acceleration):
    # TODO: Vier RK-Stufen für x, y, vx und vy berechnen
    raise NotImplementedError

## Aufgabe 3 – Erster Vergleich der Bahnen

Verwenden Sie

$$x_0=1,\quad y_0=0,\quad v_{x,0}=0,\quad v_{y,0}=0{,}8.$$

Diese Anfangsbedingung erzeugt eine deutlich elliptische Bahn. Berechnen Sie mehrere Umläufe mit allen drei Verfahren. Beschreiben Sie, ob die Bahn geschlossen bleibt, ihre Größe verändert oder ihr sonnennächster Punkt wandert.

In [ ]:
methods = {"Euler": euler_step, "Verlet": verlet_step, "RK4": rk4_step}
x0, y0, vx0, vy0 = 1.0, 0.0, 0.0, 0.8
h_trajectory = 0.03
T_trajectory = 16.0
n_trajectory = int(round(T_trajectory/h_trajectory))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), constrained_layout=True)
for ax, (name, method) in zip(axes, methods.items()):
    t, x, y, vx, vy = simulate(
        method, x0, y0, vx0, vy0, h_trajectory, n_trajectory
    )
    ax.plot(x, y, linewidth=1)
    ax.plot(0, 0, "o", color="C3", markersize=5)
    ax.set(title=name, xlabel="$x$", ylabel="$y$")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
plt.show()

**Beobachtungen:**  
TODO

## Aufgabe 4 – Fehler nach einem halben Kreis

Für

$$x_0=1,\quad y_0=0,\quad v_{x,0}=0,\quad v_{y,0}=1$$

ist die exakte Bahn ein Kreis mit Periode $2\pi$. Nach einem halben Umlauf bei $T=\pi$ gilt exakt

$$x(T)=-1,\quad y(T)=0,\quad v_x(T)=0,\quad v_y(T)=-1.$$

Berechnen Sie für $N=25,50,100,200,400$ Schritte den Fehler

$$e=\sqrt{(x_N-x(T))^2+(y_N-y(T))^2+(v_{x,N}-v_x(T))^2+(v_{y,N}-v_y(T))^2},$$

wobei $h=T/N$. Stellen Sie $e(h)$ doppelt logarithmisch dar und schätzen Sie die Ordnung aus

$$r=\log_2\!\left(\frac{e(h)}{e(h/2)}\right).$$

In [ ]:
initial_circle = (1.0, 0.0, 0.0, 1.0)
exact_half = np.array([-1.0, 0.0, 0.0, -1.0])
T_half = np.pi
n_values = 25 * 2**np.arange(5)
h_values = T_half / n_values

# TODO: Endfehler und beobachtete Ordnungen berechnen.
# TODO: Fehler doppelt logarithmisch darstellen.

## Aufgabe 5 – Energie über lange Zeiten

Vergleichen Sie für die elliptische Anfangsbedingung aus Aufgabe 3 den Energiefehler

$$E_n-E_0$$

über einen langen Zeitraum. Verwenden Sie zunächst $h=0{,}05$ und $T=400$. Da die Fehlergrößen sehr verschieden sein können, ist je ein eigenes Teilbild pro Methode sinnvoll.

Erhält RK4 die Energie exakt? Unterscheiden Sie zwischen beschränkten Schwankungen und systematischer Drift. Bedeutet eine hohe Genauigkeitsordnung automatisch gutes Langzeitverhalten?

In [ ]:
h_energy = 0.05
T_energy = 400.0
n_energy = int(round(T_energy/h_energy))

# TODO: Energiefehler für Euler, Verlet und RK4 berechnen und darstellen.

**Auswertung:**  
TODO

## Aufgabe 6 – Was geschieht bei einem anderen Potential?

Betrachten Sie statt der Gravitation das radialsymmetrische Potential

$$V_\alpha(r)=-\frac1{r^\alpha},\qquad r=\sqrt{x^2+y^2}.$$

für $\alpha \neq 1$. Was beobachten Sie?

In [ ]:
def power_law_acceleration(x, y, alpha):
    # TODO (optional): ax und ay für V_alpha berechnen
    raise NotImplementedError

**Antwort:**  
TODO